# Tunnel cost

> What would the tunnel cost? Four estimates, each worked through step by step so every assumption can be checked.

Our design is an immersed tube with ramps, on the route set out in the config notebook. **D** prices it item by item,
scaling each item of TfL's costed Silvertown immersed tube to ours. D is our working estimate. The other three are
checks:

- **A** takes TfL's own 2017 estimate for an immersed tube on this crossing and brings it up to date.
- **B** and **C** cost the alternative: a bored tunnel with lifts at each end. B builds it up item by item from
  HS2's unit rates; C scales a benchmark from another short UK tunnel.

Every figure is in £m. Each estimate gives a low, central and high case. The results are saved to
`data/derived/tunnel_cost.csv` for the viability notebook.


In [ ]:
import numpy as np
import pandas as pd
from fastcore.test import test_close

from tunnel.config import DERIVED, PRICE_YEAR, RAMP_GRADIENT, ramp_lengths, river_crossing
from tunnel.data import gdp_deflator, real

CASES = ["low", "central", "high"]

## The bored alternative

B and C cost a bored tunnel on the shortest line across, from Durand's Wharf park to the impound lock: 460 m in a
straight line, plus 10% because a bored route can't be quite straight, so 506 m. Inside it needs a 3.5 m cycle
lane beside a 2.5 m footway, so about 6 m across. Add the lining and that's about 7 m outside diameter. Bored
tunnels run deeper than immersed tubes, so it would have lifts rather than ramps.


In [ ]:
L = 460 * 1.10
OD_M = 7.0
FACE_M2 = np.pi * (OD_M / 2) ** 2
test_close(L, 506, eps=1)
test_close(FACE_M2, 38.5, eps=0.1)

## Bringing old prices up to date

The sources are priced in 2011 and 2016. We convert them to 2025 prices with the ONS GDP deflator. This is
the factor each year's prices are multiplied by:


In [ ]:
d = gdp_deflator()
{year: round(float(d[PRICE_YEAR] / d[year]), 2) for year in (2011, 2016)}

{2011: 1.46, 2016: 1.36}

B and C are built up from other projects at a stage when little has been designed. The Treasury's Green Book says
such estimates should add _optimism bias_, since early estimates almost always come in low. Its upper bounds at the
outline business case stage are 44% for standard civil engineering and 66% for non-standard. A deep river crossing
with lifts falls somewhere between, so we use 44% for the low case, 66% for the high case and the midpoint for the
central case.


In [ ]:
OPTIMISM = {"low": 0.44, "central": 0.55, "high": 0.66}

## A · TfL's estimate

TfL's _Background to Consultation Report_ (November 2017, Table 6) costs an immersed tunnel on this crossing at
£325m to £335m. An immersed tunnel is a concrete tube laid in a trench dredged across the riverbed, not a bored one,
but it crosses the same stretch of river and has to do the same job. The figure is the capital cost _plus a risk
allowance plus land and property_, in 2016 prices. It already allows for risk, so we add no optimism bias.

TfL's documents also quote £440m for the tunnel. That is a different measure, the "Net Financial Effect": whole-life
costs over 60 years, including £1.9m a year to run it, discounted to 2016. We don't use it.


In [ ]:
A = pd.Series(dict(zip(CASES, [325, 330, 335]))).map(lambda v: real(v, 2016))
A.round(0)

low        441.0
central    448.0
high       454.0
dtype: float64

## B · HS2 unit rates

HS2's _Guide to Tunnelling Costs_ gives unit rates for every part of a bored tunnel, in 2011 prices. HS2's tunnels
are bigger than ours and come in pairs, so two rates need scaling.

**Bored tunnel.** HS2's rate is £22k to £25k per metre of route for twin bores. Halve it for one bore. Excavation
scales with face area, so multiply by the ratio of our face to HS2's (8.8 m outside diameter).


In [ ]:
HS2_OD_M = 8.8
FACE_SCALE = (OD_M / HS2_OD_M) ** 2


def bored_tunnel(twin_rate_per_m, length_m=L, face_scale=FACE_SCALE):
    "£m to bore `length_m` of one tunnel, from HS2's twin-bore rate per route metre."
    return twin_rate_per_m / 2 * face_scale * length_m / 1e6

In [ ]:
test_close(bored_tunnel(23_500, length_m=1000, face_scale=1), 11.75)  # one HS2-sized bore, 1 km
round(FACE_SCALE, 2), [round(bored_tunnel(r), 1) for r in (22_000, 23_500, 25_000)]

(0.63, [3.5, 3.8, 4.0])

The excavation itself turns out to be cheap. What costs money is everything around it.

**Time on site.** Running a tunnelling site costs about £1.1m a week. We allow one year (low), eighteen months
(central) or two years (high).

**Shafts.** HS2 puts a shaft with a lift and head house at £10m to £30m. We need two, one at each portal, with lifts
big enough for bikes.

**Fit-out.** Lighting, ventilation, drainage, communications and safety systems: £4k to £6k per metre.

Each item, in 2011 £m:


In [ ]:
B_items = pd.DataFrame(
    [
        ("Tunnel boring machine", 16, 17, 18, "bought new or refurbished"),
        ("Site set-up at both portals", 35, 40, 45, "HS2 fixed cost per tunnel"),
        (
            "Bored tunnel",
            *(bored_tunnel(r) for r in (22_000, 23_500, 25_000)),
            "HS2 twin rate, halved, scaled to our face",
        ),
        ("Time on site", *(1.1 * w for w in (52, 78, 104)), "£1.1m a week for 1 to 2 years"),
        ("Shafts, lifts and entrances", *(2 * s for s in (15, 25, 30)), "two shafts at £15m to £30m each"),
        ("Fit-out", *(r * L / 1e6 for r in (4_000, 4_000, 6_000)), "£4k to £6k per metre"),
    ],
    columns=["item", *CASES, "basis"],
).set_index("item")
B_items.round(1)

,low,central,high,basis
item,,,,
Tunnel boring machine,16.0,17.0,18.0,bought new or refurbished
Site set-up at both portals,35.0,40.0,45.0,HS2 fixed cost per tunnel
Bored tunnel,3.5,3.8,4.0,"HS2 twin rate, halved, scaled to our face"
Time on site,57.2,85.8,114.4,£1.1m a week for 1 to 2 years
"Shafts, lifts and entrances",30.0,50.0,60.0,two shafts at £15m to £30m each
Fit-out,2.0,2.0,3.0,£4k to £6k per metre


Add them up, bring them to 2025 prices, and add optimism bias:


In [ ]:
B_steps = pd.DataFrame({"2011 prices": B_items[CASES].sum()})
B_steps["2025 prices"] = B_steps["2011 prices"].map(lambda v: real(v, 2011))
B_steps["with optimism bias"] = B_steps["2025 prices"] * (1 + pd.Series(OPTIMISM))
B = B_steps["with optimism bias"]
B_steps.round(0)

,2011 prices,2025 prices,with optimism bias
low,144.0,210.0,303.0
central,199.0,291.0,450.0
high,244.0,358.0,594.0


## C · A benchmark from a UK tunnel

The Infrastructure and Projects Authority and the British Tunnelling Society benchmarked UK transport tunnels in
2016/17. One short tunnel, about 0.65 km long, cost £5.63m per km of length per m² of face. Scale that to our length
and face:


In [ ]:
def benchmark_tunnel(rate=5.63, face_m2=FACE_M2, length_m=L):
    "£m for a tunnel at `rate` £m per km per m² of face."
    return rate * face_m2 * length_m / 1000

In [ ]:
test_close(benchmark_tunnel(face_m2=1, length_m=1000), 5.63)
round(benchmark_tunnel(), 1)

109.6

The benchmark covers the tunnel only, so we add B's shafts. Then bring it to 2025 prices and add optimism bias:


In [ ]:
C_steps = pd.DataFrame({"tunnel, 2025 prices": real(benchmark_tunnel(), 2016)}, index=CASES)
C_steps["+ shafts"] = C_steps["tunnel, 2025 prices"] + B_items.loc["Shafts, lifts and entrances", CASES].map(
    lambda v: real(v, 2011)
)
C_steps["with optimism bias"] = C_steps["+ shafts"] * (1 + pd.Series(OPTIMISM))
C = C_steps["with optimism bias"]
C_steps.round(0)

,"tunnel, 2025 prices",+ shafts,with optimism bias
low,149.0,193.0,277.0
central,149.0,222.0,344.0
high,149.0,237.0,393.0


## D · An immersed tube, priced from TfL's Silvertown estimate

A and D cost an immersed tube: concrete box sections cast in a dry dock, floated out and sunk into a trench dredged
across the riverbed, reaching the surface by ramps rather than lifts.

Scaling whole tunnels by size (see the check at the end) misses that much of an immersed tube's cost doesn't depend
on how big the tube is: the casting dock, bringing in marine plant, the cofferdams where the tube meets each bank,
the ramps, design and surveys. So we price each item separately.

Our source is TfL's own costed immersed tube under the Thames: the immersed tube option in the _Silvertown Crossing
Study: Tunnel Engineering_ (Mott MacDonald for TfL, 2012, Table 17.1), priced at 1 January 2012. It was a twin
two-lane road tunnel, so much bigger than ours, but its estimate is itemised and comes with unit rates and dredging
prices from contractors. For each item we work out what drives its cost, and scale it from Silvertown's tube to ours.

### The two tubes

Silvertown's immersed section was 488 m long in four 122 m elements. The report doesn't give its outside dimensions
in the text. A three-cell box about 27 m wide and 9.5 m high fits the two figures it does give: a dredged trench
65–70 m wide at the top, and £24.05m of concrete elements at £586/m³, about 84 m³ of concrete per metre.

Ours crosses about 384 m of river, in three elements of 128 m. Inside it needs a 3.5 m cycle lane beside a 2.5 m
footway, so about 7 m wide and 3.5 m high, with walls, roof and floor about 1 m thick.


In [ ]:
SILVERTOWN = {"width": 27.0, "height": 9.5, "length": 488, "elements": 4, "total_length": 488 + 552 + 341}
OURS = {"width": 7.0 + 2 * 1.0, "height": 3.5 + 2 * 1.0, "length": round(river_crossing().length), "elements": 3}


def box_concrete(width, height, wall=1.0, inner_walls=0, inner_wall=0.8):
    "Concrete per metre (m³) of a rectangular box with walls, roof and floor `wall` thick, plus any inner walls."
    return width * height - (width - 2 * wall) * (height - 2 * wall) + inner_walls * inner_wall * (height - 2 * wall)

In [ ]:
test_close(box_concrete(9, 5.5), 9 * 5.5 - 7 * 3.5)
test_close(
    box_concrete(SILVERTOWN["width"], SILVERTOWN["height"], wall=1.2, inner_walls=2), 24_049_553 / 586 / 488, eps=12
)  # about 84 m³/m

### The trench

The trench is cut to the tube's height plus cover and bedding, a couple of metres wider than the tube each side, with
sides sloping at 1 in 1.5 in the river's clay (the Silvertown study's figure). The trench is what gets dredged and
what the cofferdams at each bank have to span.


In [ ]:
def trench(width, height, cover=1.5, bedding=0.5, working=2.0, slope=1.5):
    "Cross-section area (m²) and top width (m) of the trench for a tube `width` × `height`."
    depth = height + cover + bedding
    bottom = width + 2 * working
    return bottom * depth + slope * depth**2, bottom + 2 * slope * depth

In [ ]:
s_area, s_top = trench(SILVERTOWN["width"], SILVERTOWN["height"])
o_area, o_top = trench(OURS["width"], OURS["height"])
test_close(s_top, 67.5, eps=2.5)  # the study says 65–70 m
{"Silvertown": (round(s_area), round(s_top)), "ours": (round(o_area), round(o_top))}

{'Silvertown': (555, 66), 'ours': (182, 36)}

### The ramps

The config notebook works out the ramps: the walkway climbs at 1 in 20 under the river as far as the riverbed
allows, and the rest of the climb is a ramp on land, about 250 m at the south end and 310 m at the north. TfL warns
that ramps this long may need to be shallower, with landings, to be comfortable.

Each ramp is an open concrete trough: two diaphragm walls, a base slab and inner lining walls, with the ground dug out
and taken away. It runs from ground level at the entrance down to the river wall, so on average it is half as deep as
it is at the wall. We price it with the Silvertown study's unit rates (Table 17.2), at that average depth.


In [ ]:
RATES_2012 = {"d_wall_m2": 628, "excavation_m3": 15, "disposal_m3": 40, "concrete_m3": 150, "rebar_t": 1_100}


def ramp_cost_per_m(
    depth, inside_width=OURS["width"] - 2 * 1.0, rates=RATES_2012, embed=0.5, slab=1.0, lining=0.3, rebar_per_m3=0.12
):
    "£ per metre of open ramp trough averaging `depth` deep, from unit rates."
    walls = 2 * depth * (1 + embed) * rates["d_wall_m2"]
    dig = (inside_width + 2 * 1.2) * depth * (rates["excavation_m3"] + rates["disposal_m3"])
    concrete = (inside_width + 2 * 1.2) * slab + 2 * depth * lining
    return walls + dig + concrete * (rates["concrete_m3"] + rebar_per_m3 * rates["rebar_t"])

In [ ]:
RAMPS_M = ramp_lengths()
ramps = sum(r * ramp_cost_per_m(r * RAMP_GRADIENT / 2) for r in RAMPS_M) / 1e6
f"ramps of {RAMPS_M[0]:.0f} m and {RAMPS_M[1]:.0f} m, £{ramps:.1f}m in 2012 prices"

'ramps of 247 m and 313 m, £11.7m in 2012 prices'

### Item by item

Each line takes a Silvertown item (2012 £m) and scales it by what drives its cost:

- **Tube elements:** concrete volume, at Silvertown's £586/m³.
- **Floating out and sinking:** a fixed cost per element (Silvertown: £4.6m for four).
- **Dredging and backfilling:** trench volume. Van Oord's quote includes bringing in its plant, which doesn't shrink
  with the job, so as with D's size scaling we use a power of 1 (low), 0.8 (central) or 0.6 (high).
- **Casting dock:** Silvertown's cost £48.9m, mostly retaining walls, digging and backfilling. Walls scale with the
  dock's perimeter and depth, digging with its area and depth; the river wall entrance doesn't scale. The low case
  casts the elements elsewhere and tows them in, an option the Silvertown study itself suggests. The high case pays
  Silvertown's full cost.
- **Cofferdams where the tube meets each bank:** Silvertown's Greenwich-side river works (£7.6m), scaled by trench
  width, twice.
- **Wall cladding:** Silvertown's £400/m², on 3 m of each wall.
- **Lighting, ventilation, drainage and safety systems (M&E):** per metre of tube. A foot and cycle tunnel needs far
  less than Silvertown's road tunnel (£13,911/m), so we use HS2's £4,000/m (low), £6,000/m (central) and
  Silvertown's rate (high).
- **Plant rooms at each end:** Silvertown's substations and vent buildings cost £17.6m for a road tunnel. There's no
  official figure for a foot tunnel, so we take 10%, 20% and 40% of it: a judgement.


In [ ]:
o_conc, s_conc = box_concrete(OURS["width"], OURS["height"]), 24_049_553 / 586 / SILVERTOWN["length"]
vol_ratio = (o_area * OURS["length"]) / (s_area * SILVERTOWN["length"])
area_ratio = (OURS["length"] * OURS["width"]) / (SILVERTOWN["length"] * SILVERTOWN["width"])
depth_ratio = (OURS["height"] + 6) / (SILVERTOWN["height"] + 6)  # dock depth: element height plus water to float it
basin = {
    "walls": 15.09 * area_ratio**0.5 * depth_ratio,
    "digging and backfill": (15.43 + 14.18 + 0.60) * area_ratio * depth_ratio + 0.71 * area_ratio,
    "entrance and river wall": 2.93,
}
float_out = 4.62 / SILVERTOWN["elements"] * OURS["elements"]
D_items = pd.DataFrame(
    {
        "low": [
            586 * o_conc * OURS["length"] / 1e6, float_out * 1.5, 22.45 * vol_ratio, 0,
            2 * 7.58 * o_top / s_top, 400 * 2 * 3 * OURS["length"] / 1e6, 4_000 * OURS["length"] / 1e6, 0.1 * 17.56, ramps,
        ],
        "central": [
            586 * o_conc * OURS["length"] / 1e6, float_out, 22.45 * vol_ratio**0.8, sum(basin.values()),
            2 * 7.58 * o_top / s_top, 400 * 2 * 3 * OURS["length"] / 1e6, 6_000 * OURS["length"] / 1e6, 0.2 * 17.56, ramps,
        ],
        "high": [
            586 * o_conc * OURS["length"] / 1e6, float_out, 22.45 * vol_ratio**0.6, 48.95,
            2 * 7.58 * o_top / s_top, 400 * 2 * 3 * OURS["length"] / 1e6, 13_911 * OURS["length"] / 1e6, 0.4 * 17.56, ramps,
        ],
    },
    index=[
        "tube elements", "floating out and sinking", "dredging and backfilling", "casting dock",
        "cofferdams at both banks", "wall cladding", "M&E systems", "plant rooms", "ramps",
    ],
)  # fmt: skip
D_items.round(1)

,low,central,high
tube elements,5.6,5.6,5.6
floating out and sinking,5.2,3.5,3.5
dredging and backfilling,5.8,7.6,10.0
casting dock,0.0,14.7,49.0
cofferdams at both banks,8.2,8.2,8.2
wall cladding,0.9,0.9,0.9
M&E systems,1.5,2.3,5.3
plant rooms,1.8,3.5,7.0
ramps,11.7,11.7,11.7


The low case's float-out is half as much again, for the longer tow from a dock elsewhere.

### On-costs

The Silvertown estimate adds the same on-costs to its direct works, and we use its percentages: contractor's
preliminaries (site management and facilities, 12.4% of direct works), insurances and bonds (5%), and the
contractor's overheads and profit (10%). On top of the contractor's price TfL then added its own costs: design
(10% in all), supervision (2%), site investigation (£10m) and land (£12m), plus a risk allowance from a quantified
risk assessment (15%).

We scale site investigation by length, from Silvertown's 1.4 km of tunnel and approaches to our 0.9 km including
ramps. Our land is mostly the council's park and the dock, so we take none, half or all of Silvertown's £12m: a
judgement.

**Risk.** Silvertown's own track record says a 15% risk allowance is too little. The same 2012 study costed the bored
tunnel that was actually built at £448m, including risk, design and land. The design-and-build contract let in 2019
was about £1bn, before TfL's own costs:

In [ ]:
d = gdp_deflator()
bored_2012_in_2019 = 448 * d[2019] / d[2011]
f"2012 estimate in 2019 prices £{bored_2012_in_2019:.0f}m against a contract of about £1,000m: {1000 / bored_2012_in_2019:.1f} times"

'2012 estimate in 2019 prices £513m against a contract of about £1,000m: 1.9 times'

Some of that gap is scope that changed between the study and the contract, but it is in line with how far early UK
estimates usually fall short. So instead of the study's 15%, every case gets the Green Book's optimism bias, as B
and C do: 44% (low), 55% (central) and 66% (high).


In [ ]:
def immersed_total(direct, risk, land, si):
    "Scheme cost (£m) from direct works, applying the Silvertown on-costs."
    contract = direct * 1.124 * 1.05 * 1.10
    return contract * (1 + 0.10 + 0.02) + si + land + contract * risk


our_length = OURS["length"] + sum(RAMPS_M)
si = 10 * our_length / SILVERTOWN["total_length"]
D_2012 = pd.Series(
    {
        "low": immersed_total(D_items["low"].sum(), OPTIMISM["low"], 0, si),
        "central": immersed_total(D_items["central"].sum(), OPTIMISM["central"], 6, si),
        "high": immersed_total(D_items["high"].sum(), OPTIMISM["high"], 12, si),
    }
)
test_close(immersed_total(355.05 - 16.91 - 37.30 - 10, 0.15, 12, 10), 517.8, eps=40)  # Silvertown, roughly
D = D_2012.map(lambda v: real(v, 2011))  # priced at 1 January 2012
pd.DataFrame({"direct works (2012)": D_items.sum(), "scheme (2012)": D_2012, "scheme (2025 prices)": D}).round(0)

,direct works (2012),scheme (2012),scheme (2025 prices)
low,41.0,89.0,131.0
central,58.0,139.0,203.0
high,101.0,253.0,370.0


The ramps are among the biggest items: without lifts, getting people back up from 12–16 m below the quay at the two
river walls takes about 560 m of concrete trough, longer than the tube itself. Steeper ramps, or lifts as in
estimates B and C, would change the cost and the journey time.

### Check: scaling whole tunnels by size

For comparison, here is what you get by scaling four other European immersed tubes' total costs straight down to our
size, ignoring fixed costs:

| Tunnel                       | Cost  | What the cost covers                                               | Sources                                                                                                                                                                             |
| ---------------------------- | ----- | ------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Conwy, Wales (1991)          | £102m | the tunnel. One older report says about £50m                       | [Structurae](https://structurae.net/en/structures/conwy-tunnel), [TRID](https://trid.trb.org/view/379597)                                                                           |
| Medway, England (1996)       | £80m  | the tunnel plus 800 m of dual carriageway and a junction           | [Wikipedia](https://en.wikipedia.org/wiki/Medway_Tunnel), [Structurae](https://structurae.net/en/structures/medway-tunnel)                                                          |
| Warnow, Rostock (2003)       | €220m | the tunnel, privately financed and tolled                          | [Wikipedia](https://en.wikipedia.org/wiki/Warnow_Tunnel), [Baunetz Wissen](https://www.baunetzwissen.de/gerueste-und-schalungen/objekte/verkehr/warnowquerung-bei-rostock-72526)    |
| Marieholm, Gothenburg (2020) | €170m | the design-and-build contract for the immersed section and its M&E | [Construction Index](https://www.theconstructionindex.co.uk/news/view/strabag-jv-wins-gothenburg-immersed-tunnel-contract), [TEC](https://tec-tunnel.com/projects/marieholmstunnel) |

Costs are dated to mid-construction (Marieholm to its contract), converted at that year's average exchange rate, and
brought to 2025 prices. We scale in proportion to length (to 555 m, ramp head to ramp head on TfL's route) and by
cross-section raised to a power of 1, 0.8 and 0.6.


In [ ]:
comparables = pd.DataFrame(
    [
        ("Conwy", 102, "GBP", 1989, 1060, 24.1, 10.4),
        ("Medway", 80, "GBP", 1994, 720, 23.9, 9.15),
        ("Warnow", 220, "EUR", 2001, 790, 22.5, 8.5),
        ("Marieholm", 170, "EUR", 2014, 306, 32.0, 10.0),
    ],
    columns=["tunnel", "cost_m", "currency", "price_year", "length_m", "width_m", "height_m"],
).set_index("tunnel")
GBP_PER_EUR = {2001: 0.622, 2014: 0.806}  # annual averages, approximate
comparables["cost_2025_gbp_m"] = [
    real(c * (GBP_PER_EUR[y] if cur == "EUR" else 1), y)
    for c, cur, y in zip(comparables.cost_m, comparables.currency, comparables.price_year)
]
section = comparables.width_m * comparables.height_m
naive = pd.Series(
    {
        k: (
            comparables.cost_2025_gbp_m * 555 / comparables.length_m * (OURS["width"] * OURS["height"] / section) ** p
        ).median()
        for k, p in zip(CASES, (1.0, 0.8, 0.6))
    }
)
naive.round(0)

low        38.0
central    50.0
high       66.0
dtype: float64

Scaling whole tunnels gives a few tens of millions: far less than D, because it shrinks the casting dock, marine plant,
cofferdams, ramps, design and surveys along with the tube.


## Putting them together


In [ ]:
cost = pd.DataFrame(
    {
        "A: TfL 2017, inflated": A,
        "B: HS2 unit rates": B,
        "C: benchmark + shafts": C,
        "D: immersed tube, from Silvertown": D,
    }
).T
cost.round(0)

,low,central,high
"A: TfL 2017, inflated",441.0,448.0,454.0
B: HS2 unit rates,303.0,450.0,594.0
C: benchmark + shafts,277.0,344.0,393.0
"D: immersed tube, from Silvertown",131.0,203.0,370.0


D is our working estimate: it costs the design we model, item by item. TfL's figure for an immersed tube here is
more than twice D's central case. TfL published no breakdown, so we can't say why, but its estimate also covers
land, a risk allowance and a tube of unknown size, made in 2016 before any design work. B and C show that a bored
tunnel with lifts would cost more than D's central case.


In [ ]:
COST = D
pd.concat([cost, COST.to_frame("working estimate").T]).to_csv(DERIVED / "tunnel_cost.csv")
COST.round(0)

low        131.0
central    203.0
high       370.0
dtype: float64